# 05 · Chẩn đoán và hoàn tất thí nghiệm

Chạy sau khi đã có kết quả 120 epoch của cả 6 hàm mất mát. Bốn việc:

1. Hai lượt chẩn đoán, giải thích hai hiện tượng bất thường trên đường cong
2. Hình hai bảng — hình mạnh nhất cho báo cáo
3. Ablation kiến trúc (4 lượt mới)
4. Seed thứ hai, để phát biểu được về ý nghĩa thống kê

Người phụ trách: **SV A** phần 1 và 3, **SV B** phần 2 và 4.

In [ ]:
# Chạy được cả trên Colab lẫn máy cá nhân
import sys, os
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and not Path("unet-kvasir").exists():
    # !git clone <repo cua nhom> unet-kvasir
    pass
ROOT = Path("unet-kvasir") if Path("unet-kvasir").exists() else Path("..")
os.chdir(ROOT.resolve())
sys.path.insert(0, str(Path.cwd()))

%load_ext autoreload
%autoreload 2

from src import *
print("Thư mục làm việc:", Path.cwd())
print("Thiết bị:", get_device())

## 1. Hai lượt chẩn đoán

Đường cong 120 epoch để lộ hai chuyện mà bảng số giấu đi:

- **focal dao động dữ dội rồi bị dừng sớm ở epoch 35.** Nó *không hội tụ*,
  chứ không phải hội tụ tới mức thấp. Nghi phạm là thang giá trị của loss:
  Focal nhân BCE với `alpha_t · (1-p_t)^gamma`, hệ số này chỉ khoảng 0.19 với
  pixel còn mơ hồ, nên ở cùng `lr` thì bước cập nhật hiệu dụng nhỏ hơn BCE
  khoảng 5 lần.
- **bce sụp về gần 0 nhiều lần trong 30 epoch đầu** rồi mượt hẳn về sau.
  Nghi phạm là `lr = 1e-3` hơi cao với Adam cho U-Net.

`lr` nằm trong hash `run_id` nên hai lượt dưới là hai dòng mới, không đè lên
kết quả cũ.

In [ ]:
base = Config(up_mode="transpose", skip_mode="full", epochs=120, seed=42)

diag = [
    base.replace(loss_name="focal", lr=3e-3, tag="diag_focal_lr"),
    base.replace(loss_name="bce",   lr=3e-4, tag="diag_bce_lr"),
]
for c in diag:
    print(c.run_id)

results_diag = run_sweep(diag)

**Cách đọc kết quả.** Nếu focal ở `lr=3e-3` bám được theo nhóm còn lại thì
vấn đề nằm ở thang loss chứ không phải bản thân hàm mất mát — kết luận sắc hơn
nhiều so với "Focal không hợp bài toán này". Nếu bce ở `lr=3e-4` hết sụp trong
30 epoch đầu thì xác nhận learning rate là nguyên nhân.

Chỉ khi cả hai đều không đổi mới cần thử tiếp `focal_alpha=0.5`.

## 2. Nạp lại toàn bộ đường cong từ đĩa

Không chạy lại gì. `load_history` dựng `run_id` từ Config rồi đọc file JSON mà
`run_experiment` đã ghi cạnh checkpoint.

In [ ]:
histories = {n: load_history(base.replace(loss_name=n)) for n in LOSS_NAMES}
histories["focal (lr=3e-3)"] = load_history(diag[0])
histories["bce (lr=3e-4)"]   = load_history(diag[1])

for name, h in histories.items():
    print(f"{name:20s} {len(h):3d} epoch")

Dòng nào ra `0 epoch` là do lượt đó chạy bằng bản code cũ, trước khi có
`save_history`. Chạy lại đúng lượt ấy với `skip_if_logged=False`, hoặc bỏ qua
nó khi vẽ — ô dưới đã tự bỏ qua.

## 3. Hình hai bảng

Bảng trái cho thấy cấu trúc hai nhóm và hai hiện tượng mất ổn định. Bảng phải
phóng to vùng hội tụ để thấy các đường nhóm trên quấn vào nhau — bằng chứng
trực quan cho việc không thể tuyên bố quán quân.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for name, h in histories.items():
    if not h:
        continue
    for ax in axes:
        ax.plot([x["epoch"] for x in h], [x["val_dice"] for x in h], label=name, lw=1.4)

axes[0].set_title("Toàn cảnh")
axes[1].set_title("Phóng to vùng hội tụ")
axes[1].set_xlim(80, 120); axes[1].set_ylim(0.75, 0.87)
axes[0].legend(fontsize=8, loc="lower right")
for ax in axes:
    ax.set_xlabel("epoch"); ax.set_ylabel("val Dice"); ax.grid(alpha=0.3)

fig.tight_layout()
fig.savefig(f"{base.fig_dir}/loss_sweep_120ep.png", dpi=150, bbox_inches="tight")

## 4. Ablation kiến trúc

Cấu hình `transpose` + `full` đã có trong log nên tự động bị bỏ qua — chỉ 4
lượt mới thực sự chạy, khoảng 80 phút.

In [ ]:
BEST_LOSS = "bce_dice"
abl = Config(loss_name=BEST_LOSS, epochs=120, seed=42)

up_configs   = [abl.replace(up_mode=m, skip_mode="full",      tag="ablation_up")   for m in UP_MODES]
skip_configs = [abl.replace(up_mode="transpose", skip_mode=m, tag="ablation_skip") for m in SKIP_MODES]

results_abl = run_sweep(up_configs + skip_configs)

## 5. Seed thứ hai

Chỉ chạy cho 3 cấu hình đầu bảng. Có độ lệch thì mới phát biểu được về ý nghĩa
thống kê.

In [ ]:
seed2 = [base.replace(loss_name=n, seed=1337, tag="seed2")
         for n in ("bce_dice", "bce", "weighted_bce")]
results_seed2 = run_sweep(seed2)

In [ ]:
df = RunLogger(base.log_csv).to_dataframe()
df["test_dice"] = df["test_dice"].astype(float)
df["epochs"] = df["epochs"].astype(int)
df["lr"] = df["lr"].astype(float)

m = ((df.up_mode == "transpose") & (df.skip_mode == "full")
     & (df.epochs == 120) & (df.lr == 1e-3))
agg = (df[m].groupby("loss_name")["test_dice"]
       .agg(["mean", "std", "count"]).round(4).sort_values("mean", ascending=False))
print(agg.to_markdown())

Cột `std` là thứ cho phép viết câu *"khác biệt 0.012 giữa bce_dice và bce nằm
trong nhiễu"*, hoặc bác bỏ nó nếu độ lệch hoá ra nhỏ hơn thế. `count = 1` thì
`std` là NaN — dấu hiệu cấu hình ấy chưa có seed thứ hai.

## 6. Gộp log từ hai tài khoản Colab

Phần 4 và phần 5 độc lập nhau nên chạy song song ở hai tài khoản được. Tải cả
hai file `runs.csv` về rồi gộp, khử trùng lặp theo `run_id`, giữ bản mới nhất.

In [ ]:
n = merge_logs(["logs/runs.csv", "logs/runs_may2.csv"], out_path="logs/runs.csv")
print(f"Sau khi gộp: {n} lượt chạy")

Nhớ chép cả thư mục `outputs/checkpoints/` từ máy thứ hai sang, nếu không
notebook 04 sẽ không nạp được checkpoint để vẽ lưới ảnh so sánh.